<a href="https://colab.research.google.com/github/maryamfarhan/chaosbench-colab/blob/main/ChaosBench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!nproc
!python --version

Fri Nov 13 11:55:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/leap-stc/ChaosBench.git
%cd ChaosBench
!sed -E 's/==.*//' requirements.txt > req_unpinned.txt
!pip install -r req_unpinned.txt
!pip install zarr


fatal: destination path 'ChaosBench' already exists and is not an empty directory.
/content/ChaosBench


In [5]:
!df -h /content
%cd /content/ChaosBench
!mkdir -p data
%cd data
!wget https://huggingface.co/datasets/LEAP/ChaosBench/resolve/main/process.sh
!chmod +x process.sh

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  592M  113G   1% /
/content/ChaosBench
/content/ChaosBench/data
--2026-11-13 11:58:06--  https://huggingface.co/datasets/LEAP/ChaosBench/resolve/main/process.sh
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/LEAP/ChaosBench/9243dc670d85433b8c757c9608b21a52eb37ad44/process.sh?%2Fdatasets%2FLEAP%2FChaosBench%2Fresolve%2Fmain%2Fprocess.sh=&etag=%2253875ee5028d6f050584ad2bd6427987cff8791b%22 [following]
--2026-11-13 11:58:06--  https://huggingface.co/api/resolve-cache/datasets/LEAP/ChaosBench/9243dc670d85433b8c757c9608b21a52eb37ad44/process.sh?%2Fdatasets%2FLEAP%2FChaosBench%2Fresolve%2Fmain%2Fprocess.sh=&etag=%2253875ee5028d6f050584ad2bd6427987cff8791b%22
Reusing existing connection to huggingfa

In [6]:
%cd /content/ChaosBench/data
!apt-get install -y pigz > /dev/null 2>&1
!nohup ./process.sh era5 with_pigz > era5_log.txt 2>&1 &

/content/ChaosBench/data


In [18]:
!tail -c 400 /content/ChaosBench/data/era5_log.txt
!df -h /content | tail -1
!ls -lh /content/ChaosBench/data

.ac
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.55, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-11-13 12:05:51 ERROR 404: Not Found.

COMBINING CHUNKS, THIS MAY TAKE A WHILE...
EXTRACTING FOLDER, THIS MAY TAKE A WHILE...
Use pigz for parallel decompression...
overlay         113G   51G   62G  46% /
total 68M
drwxr-sr-x 16438 548497 55300 932K Jan 25  2024 era5
-rw-r--r--     1 root   root   67M Nov 13 12:13 era5_log.txt
-rwxr-xr-x     1 root   root  1009 Nov 13 11:43 process.sh


In [19]:
!while [ ! -d /content/ChaosBench/data/era5 ]; do sleep 30; done; echo "ERA5 ready"

ERA5 ready


In [20]:
%%writefile /content/ChaosBench/chaosbench/configs/unet_test_s2s.yaml
model_args:
    model_name: 'unet_s2s'
    input_size: 60
    output_size: 60
    learning_rate: 0.01
    num_workers: 2
    epochs: 1
    t_max: 1
    only_headline: False

data_args:
    batch_size: 4
    train_years: [1979, 1980, 1981]
    val_years: [2016]
    n_step: 1
    lead_time: 1
    land_vars: []
    ocean_vars: []

Overwriting /content/ChaosBench/chaosbench/configs/unet_test_s2s.yaml


In [21]:
!cd /content/ChaosBench/data && ./process.sh climatology with_pigz > climatology_log.txt 2>&1

In [22]:
!ls /content/ChaosBench/data/era5 | wc -l
!du -sh /content/ChaosBench/data/era5
!ls /content/ChaosBench/data/climatology

16436
51G	/content/ChaosBench/data/era5
climatology_cma.zarr	       climatology_lra5.zarr
climatology_ecmwf.zarr	       climatology_ncep.zarr
climatology_era5_spatial.zarr  climatology_oras5_spatial.zarr
climatology_era5.zarr	       climatology_oras5.zarr
climatology_lra5_spatial.zarr  climatology_ukmo.zarr


In [23]:
!cd /content/ChaosBench && python train.py --config_filepath chaosbench/configs/unet_test_s2s.yaml

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
I0000 00:00:1794573165.075117   13728 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794573168.766848   13728 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
┏━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ UNet │ 31.1 M │ train │     0 │
│ 1 │ loss  │ MSE  │  